# 13.03 Google — 공급자 카탈로그

Google 통합은 두 패키지로 나뉜다.

- **`langchain-google-genai`** — Gemini API 직결 (`ChatGoogleGenerativeAI`, `GoogleGenerativeAIEmbeddings`). v4 부터 `vertexai=True` 로 Vertex AI 모드도 단일 진입점에서 라우팅.
- **`langchain-google-community`** — Vertex AI Search · GCS · Drive · YouTube · BigQuery 등 GCP 데이터 통합 묶음.

## 학습 목표

- 두 패키지의 역할 분담과 모델 ID(`gemini-2.5-flash` / `gemini-2.5-pro`) 명명 규칙
- Gemini API vs Vertex AI 라우팅: `vertexai=True, project=..., location="us-central1"`
- 공급자 1순위 차별 기능: **Vertex AI Search** (GCP 의 매니지드 retrieval)
- Embedding `task_type` 5종 — query/document/similarity/classification/clustering

## 13.03.1 환경 설정

필요 패키지: `langchain-google-genai` (필수) + `langchain-google-community` (선택, Vertex Search · GCS 등 사용 시).
`.env` 에 `GOOGLE_API_KEY` 가 있어야 한다 (Gemini API 직결). Vertex AI 모드는 `gcloud auth application-default login` 으로 ADC 자격증명 발급.

```bash
uv pip install langchain-google-genai langchain-google-community
```

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
assert os.environ["GOOGLE_API_KEY"], "GOOGLE_API_KEY 필요"

## 13.03.2 공급자 제품군 전체 지도

| 카테고리 | 심볼 | 패키지 | 카테고리 노트북 |
|---------|------|--------|----------------|
| Chat | `ChatGoogleGenerativeAI` | `langchain-google-genai` | [`01_chat_models/03_google.ipynb`](../01_chat_models/03_google.ipynb) |
| Embeddings | `GoogleGenerativeAIEmbeddings` | `langchain-google-genai` | [`02_embeddings/02_google.ipynb`](../02_embeddings/02_google.ipynb) |
| Cloud Storage loader | `GCSDirectoryLoader` · `GCSFileLoader` | `langchain-google-community` | [`04_document_loaders/03_cloud_storage.ipynb`](../04_document_loaders/03_cloud_storage.ipynb) |
| Drive loader | `GoogleDriveLoader` | `langchain-google-community` | [`04_document_loaders/04_productivity.ipynb`](../04_document_loaders/04_productivity.ipynb) |
| Vertex AI Search retriever | `VertexAISearchRetriever` | `langchain-google-community` | [`05_retrievers/05_vendor_managed.ipynb`](../05_retrievers/05_vendor_managed.ipynb) |
| BigQuery vector store | `BigQueryVectorStore` | `langchain-google-community` | — |
| Web search 도구 | `GoogleSearchAPIWrapper` | `langchain-google-community` | — |

v4(`langchain-google-genai>=4.0`) 에서는 별도 `langchain-google-vertexai` 패키지 없이 `ChatGoogleGenerativeAI(vertexai=True, project=..., location=...)` 로 Vertex 라우팅이 가능해졌다.

## 13.03.3 기본 사용 — chat

최소 한 줄.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", max_output_tokens=256)
print(model.invoke("한국어로 한 문장만: 너는 누구냐?").content)

## 13.03.4 공급자 특화 기능 — Vertex AI Search

Google 의 1순위 차별 기능은 **Vertex AI Search** 다. GCP 가 호스팅하는 매니지드 retrieval — 문서 인덱싱·청킹·하이브리드 검색·랭킹을 모두 GCP 측에서 처리한다.
LangChain 에서는 `langchain-google-community` 의 `VertexAISearchRetriever` 로 노출.

```python
from langchain_google_community import VertexAISearchRetriever

retriever = VertexAISearchRetriever(
    project_id='my-gcp-project',
    location_id='global',
    data_store_id='my-corpus',
    max_documents=5,
)
docs = retriever.invoke('환불 정책')
```

데이터 스토어는 GCP 콘솔에서 GCS/Drive/HTML/구조화 데이터를 가리켜 만든다. 한국어 포함 다국어 검색 품질이 강점.
상세는 [`05_retrievers/05_vendor_managed.ipynb`](../05_retrievers/05_vendor_managed.ipynb).

또 다른 차별점: **Gemini 의 1M~2M 토큰 컨텍스트** + **YouTube/이미지/오디오 멀티모달 입력**.

In [ ]:
# 패키지가 설치돼 있다면 시그니처만 검증 — 실제 데이터스토어가 없으므로 호출은 하지 않는다.
import importlib, inspect
spec = importlib.util.find_spec("langchain_google_community")
if spec is not None:
    from langchain_google_community import VertexAISearchRetriever
    print("VertexAISearchRetriever fields:",
          [f for f in VertexAISearchRetriever.model_fields if not f.startswith("_")][:15])
else:
    print('langchain-google-community 미설치 — `uv pip install langchain-google-community` 로 설치하면 VertexAISearchRetriever 사용 가능')

## 13.03.5 가격 · 한도 · 리전

(USD / 1M tokens, 2026-04 시점 콘솔 표시가의 대략적 위치)

| 모델 | 컨텍스트 | 입력 | 출력 | 비고 |
|------|---------|-----|------|------|
| `gemini-2.5-pro` | 2M | 중상 | 높음 | 최상급, 멀티모달, 긴 컨텍스트 |
| `gemini-2.5-flash` | 1M | 저렴 | 저렴 | 에이전트 기본값, 멀티모달 |
| `gemini-2.5-flash-lite` | 1M | 매우 저렴 | 매우 저렴 | 분류·라우팅 |
| `text-embedding-004` | 2k | 저렴 | — | 768d, `task_type` 5종 |
| `gemini-embedding-001` | 8k | 저렴 | — | 3072d (`output_dimensionality` 로 축소 가능) |

리전:

- Gemini API: 글로벌 단일 엔드포인트 (us 기반)
- Vertex AI: `us-central1`, `europe-west4`, `asia-northeast1`(도쿄), `asia-northeast3`(서울) 등 — 데이터 거주 요구 시 Vertex 선택

한도: 무료 Tier 는 분당 RPM 이 낮다 (15 RPM 안팎). 결제 활성 시 Tier 1 부터 RPM/TPM 이 크게 늘어난다.

## 핵심 정리

- **chat 1순위**: 일반 = `gemini-2.5-flash`, 멀티모달/긴 컨텍스트 = `pro`, 라우팅 = `flash-lite`
- **embedding 1순위**: 다국어 = `text-embedding-004` + `task_type="retrieval_document"`/`"retrieval_query"` 분리
- **공급자 차별 1순위**: Vertex AI Search — GCP 데이터 거버넌스 안에서 RAG 를 통째 매니지드로 돌리고 싶을 때
- **Vertex 모드**는 `vertexai=True, project=..., location=...` 한 패키지로 — v4 이전 `langchain-google-vertexai` 는 deprecated

## 다음

- [`01_chat_models/03_google.ipynb`](../01_chat_models/03_google.ipynb) — ChatGoogleGenerativeAI 깊은 사용
- [`02_embeddings/02_google.ipynb`](../02_embeddings/02_google.ipynb) — `task_type` 5종 사용 패턴
- [`05_retrievers/05_vendor_managed.ipynb`](../05_retrievers/05_vendor_managed.ipynb) — Vertex AI Search retriever
- [`04_document_loaders/03_cloud_storage.ipynb`](../04_document_loaders/03_cloud_storage.ipynb) — GCS / Drive 로더

## 참고

- `langchain-google-genai` PyPI: https://pypi.org/project/langchain-google-genai/
- `langchain-google-community` PyPI: https://pypi.org/project/langchain-google-community/
- 공식 docs: https://docs.langchain.com/oss/python/integrations/providers/google
- Vertex AI Search: https://cloud.google.com/generative-ai-app-builder/docs/introduction